# Actor Holdout Feature Ablation

Ce notebook reprend le script `actor_holdout_feature_ablation.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Ablation de features pour reduire les echecs actor-holdout du modele live.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Actor-held-out feature/regularization ablation for sequence danger models.
- Artefacts controles : Actor-held-out feature mitigation ablation exists. (`runs/exp_035_actor_holdout_feature_ablation/metrics/actor_holdout_feature_summary.csv`); Actor-held-out feature architecture extension exists. (`runs/exp_036_actor_holdout_feature_arch_extension/metrics/actor_holdout_feature_summary.csv`).
- Run par defaut : `runs/exp_035_actor_holdout_feature_ablation`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "actor_holdout_feature_ablation.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

from actor_holdout_sequence_experiments import selection_score
from ml_pipeline import ROOT, write_json
from sequence_experiments import append_report, evaluate_catalogue_model, make_run_dir, train_one_model
from sequence_feature_ablation_experiments import build_feature_groups, load_sequence


## Fonction `make_actor_split`

Cette cellule definit `make_actor_split`. Elle prepare une partie du script.

In [ ]:
def make_actor_split(meta, held_actor, seed):
    videos = meta.groupby("video_id", as_index=False).agg(actor=("actor", "first"), is_danger_clip=("is_danger_clip", "max"))
    test_ids = videos[videos["actor"] == held_actor]["video_id"].to_numpy()
    rest = videos[videos["actor"] != held_actor].copy()
    stratify = rest["is_danger_clip"].astype(int).to_numpy()
    if len(np.unique(stratify)) < 2 or min(np.bincount(stratify)) < 2:
        stratify_arg = None
    else:
        stratify_arg = stratify
    train_ids, val_ids = train_test_split(
        rest["video_id"].to_numpy(),
        test_size=0.25,
        random_state=seed,
        stratify=stratify_arg,
    )
    split = {video_id: "train" for video_id in train_ids}
    split.update({video_id: "val" for video_id in val_ids})
    split.update({video_id: "test" for video_id in test_ids})
    return split


## Fonction `train_normalize`

Cette cellule definit `train_normalize`. Elle prepare une partie du script.

In [ ]:
def train_normalize(X_raw, meta):
    train_mask = meta["split"].to_numpy() == "train"
    flat = X_raw[train_mask].reshape(-1, X_raw.shape[-1])
    mean = flat.mean(axis=0)
    std = flat.std(axis=0)
    std = np.where(std > 1e-6, std, 1.0)
    return ((X_raw - mean) / std).astype(np.float32), mean.astype(np.float32), std.astype(np.float32)


## Fonction `summarize`

Cette cellule definit `summarize`. Elle prepare une partie du script.

In [ ]:
def summarize(metrics, run_dir):
    h1 = metrics[metrics["horizon_s"].astype(float).eq(1.0)].copy()
    rows = []
    for (held_actor, feature_set, spec, split), group in h1.groupby(["held_actor", "feature_set", "spec", "split"]):
        row = group.iloc[0]
        rows.append(
            {
                "held_actor": held_actor,
                "feature_set": feature_set,
                "spec": spec,
                "split": split,
                "feature_count": int(row["feature_count"]),
                "average_precision": float(row["average_precision"]),
                "roc_auc": row["roc_auc"],
                "hit_rate": row.get("best_hit_rate"),
                "false_alarms_per_min": row.get("best_false_alarms_per_min"),
                "window_precision": row.get("best_window_precision"),
                "window_recall": row.get("best_window_recall"),
                "selection_score": float(row["selection_score"]) if pd.notna(row["selection_score"]) else None,
            }
        )
    summary = pd.DataFrame(rows)
    summary.to_csv(run_dir / "metrics" / "actor_holdout_feature_summary.csv", index=False)

    lines = ["# Actor-Held-Out Feature/Regularization Ablation", ""]
    lines.append("Each actor is held out as test once. The remaining actor's parent videos are split into train/val. Feature groups are selected before train-only normalization.")
    lines.append("")
    for held_actor in sorted(summary["held_actor"].unique()):
        lines.append(f"## Held Out: {held_actor}")
        lines.append("")
        test = summary[(summary["held_actor"] == held_actor) & (summary["split"] == "test")].sort_values("average_precision", ascending=False)
        lines.append("| rank | feature set | spec | features | AP | ROC AUC | hit | FA/min | precision |")
        lines.append("|---:|---|---|---:|---:|---:|---:|---:|---:|")
        for rank, (_, row) in enumerate(test.iterrows(), start=1):
            lines.append(
                f"| {rank} | {row['feature_set']} | {row['spec']} | {int(row['feature_count'])} | "
                f"{row['average_precision']:.3f} | {float(row['roc_auc']):.3f} | {float(row['hit_rate']):.3f} | "
                f"{float(row['false_alarms_per_min']):.3f} | {float(row['window_precision']):.3f} |"
            )
        lines.append("")
    lines.append("## Interpretation")
    lines.append("")
    lines.append("- This is a mitigation audit for the actor gap, not a replacement for the repeated parent-split catalogue.")
    lines.append("- If a reduced feature group improves held-out actor AP, it suggests some all-feature models were using actor/staging-specific cues.")
    lines.append("- If no feature group closes the gap, the honest conclusion remains that the current two-actor dataset cannot prove actor robustness.")
    (run_dir / "actor_holdout_feature_summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    return summary


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    source, X_norm, y, base_meta, feature_columns = load_sequence(args.sequence_run)
    data = np.load(source / "features" / "sequence_dataset.npz")
    mean = data["mean"].astype(np.float32)
    std = data["std"].astype(np.float32)
    X_raw_all = X_norm * std.reshape(1, 1, -1) + mean.reshape(1, 1, -1)
    videos = pd.read_csv(ROOT / "annotations" / "videos.csv")[["video_id", "actor"]]
    base_meta = base_meta.merge(videos, on="video_id", how="left")
    actors = [actor for actor in sorted(base_meta["actor"].dropna().unique()) if actor]
    groups = build_feature_groups(feature_columns)
    if args.groups:
        groups = {name: groups[name] for name in args.groups if name in groups}
    specs = [
        ("tcn_aug_bce", "tcn", True, "bce"),
        ("tcn_noaug_bce", "tcn", False, "bce"),
        ("tcn_aug_focal", "tcn", True, "focal"),
    ]
    if args.include_cnn1d:
        specs.append(("cnn1d_aug_focal", "cnn1d", True, "focal"))
    if args.include_recurrent:
        specs.extend(
            [
                ("gru_noaug_bce", "gru", False, "bce"),
                ("lstm_aug_bce", "lstm", True, "bce"),
                ("cnn_gru_aug_bce", "cnn_gru", True, "bce"),
            ]
        )

    run_dir = make_run_dir(args.run_name)
    write_json(
        run_dir / "config.json",
        {
            "source_run": str(source),
            "actors": actors,
            "feature_groups": {name: [feature_columns[i] for i in idxs] for name, idxs in groups.items()},
            "specs": [spec[0] for spec in specs],
            "epochs": args.epochs,
            "patience": args.patience,
            "split_policy": "held actor is test; remaining actor videos split into train/val; feature groups train-normalized after split",
        },
    )
    device = torch.device("cuda" if torch.cuda.is_available() and args.device == "auto" else args.device)
    all_metrics = []
    all_history = []
    split_rows = []
    for held_actor in actors:
        split = make_actor_split(base_meta, held_actor, args.seed)
        meta = base_meta.copy()
        meta["split"] = meta["video_id"].map(split)
        meta.to_csv(run_dir / "features" / f"actor_holdout_feature_{held_actor}_index.csv", index=False)
        split_counts = meta.groupby("split")["video_id"].nunique().to_dict()
        split_rows.append({"held_actor": held_actor, **split_counts})
        for group_name, idxs in groups.items():
            X_group_raw = X_raw_all[:, :, idxs].astype(np.float32)
            X_group, split_mean, split_std = train_normalize(X_group_raw, meta)
            np.savez_compressed(
                run_dir / "features" / f"normalizer_holdout_{held_actor}_{group_name}.npz",
                mean=split_mean,
                std=split_std,
            )
            for spec_name, kind, augment, loss in specs:
                model_name = f"holdout_{held_actor}_{group_name}_{spec_name}"
                print(f"training {model_name} ({len(idxs)} features)")
                model_args = SimpleNamespace(
                    seed=args.seed,
                    batch_size=args.batch_size,
                    lr=args.lr,
                    weight_decay=args.weight_decay,
                    epochs=args.epochs,
                    patience=args.patience,
                    loss=loss,
                    label_smoothing=args.label_smoothing,
                    focal_gamma=args.focal_gamma,
                )
                model, history, train_time_s, model_size_bytes = train_one_model(
                    model_name,
                    kind,
                    augment,
                    X_group,
                    y,
                    meta,
                    run_dir,
                    model_args,
                    device,
                )
                for row in history:
                    row["held_actor"] = held_actor
                    row["feature_set"] = group_name
                    row["feature_count"] = len(idxs)
                    row["spec"] = spec_name
                all_history.extend(history)
                rows, _ = evaluate_catalogue_model(
                    model_name,
                    model,
                    X_group,
                    y,
                    meta,
                    run_dir,
                    device,
                    train_time_s,
                    model_size_bytes,
                    args.batch_size,
                    loss,
                )
                for row in rows:
                    row["held_actor"] = held_actor
                    row["feature_set"] = group_name
                    row["feature_count"] = len(idxs)
                    row["spec"] = spec_name
                    row["selection_score"] = selection_score(row) if row["split"] == "val" and float(row["horizon_s"]) == 1.0 else np.nan
                all_metrics.extend(rows)
                pd.DataFrame(all_metrics).to_csv(run_dir / "metrics" / "actor_holdout_feature_metrics.csv", index=False)
                pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "actor_holdout_feature_training_history.csv", index=False)
    metrics = pd.DataFrame(all_metrics)
    metrics.to_csv(run_dir / "metrics" / "actor_holdout_feature_metrics.csv", index=False)
    pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "actor_holdout_feature_training_history.csv", index=False)
    pd.DataFrame(split_rows).to_csv(run_dir / "metrics" / "actor_holdout_feature_split_counts.csv", index=False)
    summarize(metrics, run_dir)
    append_report(
        run_dir,
        "Actor-Held-Out Feature Ablation Completion",
        f"- Actors: `{actors}`\n- Feature groups: `{list(groups)}`\n- Specs: `{[spec[0] for spec in specs]}`\n- Summary: `{run_dir / 'actor_holdout_feature_summary.md'}`",
    )
    print(run_dir)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Actor-held-out feature/regularization ablation for sequence danger models.")
    parser.add_argument("--sequence-run", default="runs/exp_008_sequence_len60_catalogue")
    parser.add_argument("--run-name", default="exp_035_actor_holdout_feature_ablation")
    parser.add_argument(
        "--groups",
        nargs="*",
        default=["all_features", "zone_motion_only", "hands_arms_geometry_motion", "head_torso_geometry_motion", "no_zone_geometry"],
    )
    parser.add_argument("--include-cnn1d", action="store_true")
    parser.add_argument("--include-recurrent", action="store_true")
    parser.add_argument("--epochs", type=int, default=14)
    parser.add_argument("--patience", type=int, default=3)
    parser.add_argument("--batch-size", type=int, default=128)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--weight-decay", type=float, default=2e-4)
    parser.add_argument("--label-smoothing", type=float, default=0.08)
    parser.add_argument("--focal-gamma", type=float, default=2.0)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--device", default="auto")
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_035_actor_holdout_feature_ablation_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["actor_holdout_feature_ablation.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
